# 🎯 Module 08: Model Evaluation

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 08 — Model Evaluation  
> **Date:** 17 January 2026  
> **Previous:** 07 — Dimensionality Reduction  
> **Next:** 09 — Hyperparameter Tuning

---

A model isn't useful just because it makes predictions — we need to know **how well it performs on unseen data**.

In this notebook we'll cover:

- **Confusion matrix** (TP, TN, FP, FN)
- **Accuracy, Precision, Recall, F1**
- **ROC-AUC**
- **Precision-Recall curve**
- **Cross-validation** strategies
- **Learning curves**
- **Data leakage** and how to avoid it
- A complete evaluation workflow

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Read a **confusion matrix** and understand its four cells
- Compute and interpret **accuracy, precision, recall, F1**
- Choose the **right metric** for a business problem
- Build and interpret **ROC curves** and **PR curves**
- Compare **ROC-AUC** vs **PR-AUC** for imbalanced data
- Apply **K-Fold**, **Stratified K-Fold**, and **Time Series Split**
- Read a **learning curve** and diagnose under/overfitting
- Detect and prevent **data leakage**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    StratifiedKFold, KFold, TimeSeriesSplit,
    learning_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    classification_report,
)

np.random.seed(42)
print('✅ Setup ready.')

---
## 🎯 1. Confusion Matrix

A confusion matrix compares **actual vs predicted classes**.

| | Predicted Positive | Predicted Negative |
| --- | ---: | ---: |
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

### Terms

- **TP — True Positive:** correctly predicted positive
- **TN — True Negative:** correctly predicted negative
- **FP — False Positive:** predicted positive, actually negative
- **FN — False Negative:** predicted negative, actually positive

### Spam detector example

```text
TP → spam correctly detected
TN → normal message correctly detected
FP → normal message incorrectly marked spam
FN → spam incorrectly classified as normal
```

---
## 📊 2. Classification Metrics

### Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Good when classes are reasonably balanced.

### Precision
$$\text{Precision} = \frac{TP}{TP + FP}$$

Of everything predicted positive, how many were actually positive?  
**Useful when false positives are expensive.**

### Recall
$$\text{Recall} = \frac{TP}{TP + FN}$$

Of all actual positives, how many did we detect?  
**Useful when false negatives are expensive.**

### F1 Score
$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Balance** between precision and recall.

In [ ]:
# Load a binary classification dataset
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# A reasonable model
model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=200, random_state=42)),
])
model.fit(X_train, y_train)

# Predictions + probabilities
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:')
print(f'              Pred Neg   Pred Pos')
print(f'Actual Neg     {cm[0,0]:>5}      {cm[0,1]:>5}')
print(f'Actual Pos     {cm[1,0]:>5}      {cm[1,1]:>5}')

In [ ]:
# All metrics at once
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

---
## 📈 3. ROC Curve & AUC

**ROC = Receiver Operating Characteristic**

Plots across all classification thresholds:

- **TPR (True Positive Rate)** = Recall
  $$TPR = \frac{TP}{TP + FN}$$
- **FPR (False Positive Rate)**
  $$FPR = \frac{FP}{FP + TN}$$

### AUC

**AUC = Area Under the ROC Curve** — summarizes the curve into a value in [0, 1].

| AUC | Interpretation |
| ---: | --- |
| 1.0 | Perfect separation |
| 0.9 | Very strong |
| 0.7 | Moderate |
| 0.5 | Random |
| < 0.5 | Worse than random |

AUC measures how well the model **ranks positives above negatives**.

In [ ]:
# ROC curve + AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label=f'LogReg (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'ROC-AUC: {auc:.4f}')

---
## 🔍 4. Precision-Recall Curve

**Precision-Recall (PR) curve** plots precision vs recall at different thresholds.

### ⚠️ Especially useful for imbalanced datasets

Example:

```text
100,000 transactions
99,000 legitimate
1,000 fraudulent
```

A model predicting **everything as legitimate**:

```text
Accuracy = 99%   ← sounds great!
Fraud detected = 0
```

**ROC can be too optimistic on imbalanced data. PR curve is better.**

### When to use what

| Situation | Use |
| --- | --- |
| Balanced classification | ROC-AUC |
| Highly imbalanced | PR curve / PR-AUC |
| FP expensive | Precision |
| FN expensive | Recall |
| Balance needed | F1 |
| Overall correctness | Accuracy |

In [ ]:
# PR curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, linewidth=2, label=f'LogReg (AP = {ap:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Average Precision (AP): {ap:.4f}')

---
## ⚠️ 5. Why Accuracy Can Be Misleading

On imbalanced data, a **lazy classifier** can look great on accuracy but be **useless**.

Let's prove it.

In [ ]:
# Simulate a heavily imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=10000, n_features=20,
    weights=[0.99, 0.01], flip_y=0,
    random_state=42,
)
print('Class distribution:', np.bincount(y_imb).tolist())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

# Dummy classifier: always predict the majority class
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
dummy_preds = dummy.predict(X_te)

# Real classifier with balanced class weights
lr_imb = LogisticRegression(max_iter=200, class_weight='balanced', random_state=42)
lr_imb.fit(X_tr, y_tr)
lr_preds = lr_imb.predict(X_te)

print(f'\n{"Model":<22} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 66)
for name, p in [('Dummy (majority)', dummy_preds), ('LogReg (balanced)', lr_preds)]:
    print(f'{name:<22} '
          f'{accuracy_score(y_te, p):>10.3f} '
          f'{precision_score(y_te, p, zero_division=0):>10.3f} '
          f'{recall_score(y_te, p):>10.3f} '
          f'{f1_score(y_te, p):>10.3f}')

---
## 🔄 6. Cross-Validation

A single train/test split can be **misleading** — it depends on **which** samples happen to be in each set.

**Cross-validation** evaluates the model on multiple splits.

### K-Fold (K = 5)

```text
Fold 1 → Validation
Fold 2 → Validation
Fold 3 → Validation
Fold 4 → Validation
Fold 5 → Validation
```

Each fold becomes the validation set once; the final score is the **average**.

### Common strategies

| Strategy | Use |
| --- | --- |
| K-Fold | General datasets |
| Stratified K-Fold | Classification (preserves class ratio) |
| Leave-One-Out | Very small datasets |
| Time Series Split | Time-dependent data |
| Group K-Fold | Grouped / dependent samples |

In [ ]:
# Compare plain K-Fold with Stratified K-Fold on the breast cancer data
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=200, random_state=42)),
])

scores_kf = cross_val_score(pipe, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='accuracy')
scores_skf = cross_val_score(pipe, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='accuracy')

print(f'K-Fold           scores: {scores_kf.round(4)} | mean = {scores_kf.mean():.4f} ± {scores_kf.std():.4f}')
print(f'Stratified K-Fold scores: {scores_skf.round(4)} | mean = {scores_skf.mean():.4f} ± {scores_skf.std():.4f}')

In [ ]:
# Multi-metric cross-validation
results = cross_validate(
    pipe, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
)

print('Cross-validation results (mean ± std):')
for metric in ['test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']:
    scores = results[metric]
    print(f'  {metric:<18} {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# TimeSeriesSplit demo on a synthetic time series
n = 100
X_ts = np.arange(n).reshape(-1, 1)
y_ts = (np.sin(np.arange(n) / 5) > 0).astype(int)

tscv = TimeSeriesSplit(n_splits=5)
print('TimeSeriesSplit folds (train_size, val_size):')
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_ts)):
    print(f'  Fold {i+1}: train={len(tr_idx):>3}  val={len(val_idx):>3}  (val covers indices {val_idx[0]}–{val_idx[-1]})')

---
## 📉 7. Learning Curves

A **learning curve** shows model performance as the amount of **training data increases**.

Usually we compare:

```text
Training Score
Validation Score
```

against:

```text
Training Set Size
```

### Diagnosing problems

| Symptom | Diagnosis |
| --- | --- |
| High train, low val | **Overfitting** |
| Both low | **Underfitting** |
| Both high and close | **Good generalization** |

In [ ]:
# Learning curve for a complex model (overfit-prone)
def plot_learning_curve(model, X, y, title, cv=5):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy',
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, 'o-', label='Training score')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
    plt.plot(train_sizes, val_mean, 's-', label='Validation score')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)
    plt.xlabel('Training set size'); plt.ylabel('Accuracy')
    plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

# Complex model → overfit
plot_learning_curve(
    DecisionTreeClassifier(random_state=42),
    X, y, 'Learning Curve — Deep Decision Tree (overfitting)')

In [ ]:
# Simple model — underfitting
plot_learning_curve(
    LogisticRegression(max_iter=200, random_state=42),
    X, y, 'Learning Curve — Logistic Regression (well-fit)',
    cv=5,
)

---
## 🚫 8. Avoid Data Leakage

Evaluation must represent how the model behaves on **unseen real-world data**.

### ❌ Wrong
```python
X = scaler.fit_transform(X)        # scales ENTIRE dataset including test
X_train, X_test = train_test_split(X, ...)
```

### ✅ Correct
```python
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
```

### ✅ Best — use a Pipeline
```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
```

The pipeline keeps preprocessing **inside** the evaluation process — no leakage by construction.

In [ ]:
# Show the difference: leaky scaling vs correct
X_raw, y_raw = load_breast_cancer(return_X_y=True)

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

# ❌ Leaky: fit on entire dataset
leak_scaler = StandardScaler().fit(X_raw)
X_tr_leak = leak_scaler.transform(X_tr)
X_te_leak = leak_scaler.transform(X_te)
m_leak = LogisticRegression(max_iter=200).fit(X_tr_leak, y_tr)
acc_leak = m_leak.score(X_te_leak, y_te)

# ✅ Correct: fit only on train
ok_scaler = StandardScaler().fit(X_tr)
X_tr_ok = ok_scaler.transform(X_tr)
X_te_ok = ok_scaler.transform(X_te)
m_ok = LogisticRegression(max_iter=200).fit(X_tr_ok, y_tr)
acc_ok = m_ok.score(X_te_ok, y_te)

print(f'Leaky     accuracy: {acc_leak:.4f}')
print(f'Correct   accuracy: {acc_ok:.4f}')
print('\nDifference is small on a clean dataset. On messy real-world data, the difference can be huge.')

---
## 🧪 9. Practical Evaluation Workflow

```text
Raw Dataset
     ↓
Train / Test Split
     ↓
Cross-Validation on Training Data
     ↓
Train Model
     ↓
Tune / Select Model
     ↓
Evaluate on Test Set
     ↓
Confusion Matrix / Metrics
     ↓
ROC-AUC / PR Curve
     ↓
Learning Curve
     ↓
Final Model
```

> **The test set stays untouched until final evaluation.**

---
## 📝 10. Practice Checklist

- [ ] Create a classification model using a scikit-learn dataset
- [ ] Generate a confusion matrix
- [ ] Calculate accuracy, precision, recall, F1
- [ ] Plot an ROC curve and calculate ROC-AUC
- [ ] Plot a Precision-Recall curve
- [ ] Compare 5-fold and Stratified K-Fold cross-validation
- [ ] Generate a learning curve
- [ ] Create an imbalanced example and explain why accuracy can mislead
- [ ] Identify overfitting or underfitting from a learning curve

---
## 🏋️ 11. Hands-On Exercises

### Exercise 1 — Manual Metrics
Given `TP=80, FP=15, FN=5, TN=100`, compute by hand:
- Accuracy
- Precision
- Recall
- F1

Then verify with scikit-learn.

### Exercise 2 — Threshold Sweep
For a trained model, try thresholds `t ∈ {0.1, 0.3, 0.5, 0.7, 0.9}`. Report precision/recall/F1 at each. Which threshold gives the best F1?

### Exercise 3 — Class Imbalance
Create an imbalanced dataset (1:50 ratio). Compare a plain classifier vs one with `class_weight='balanced'`. Which metric improves most?

### Exercise 4 — Multiple Models
Evaluate Logistic Regression, Random Forest, and SVM on the same dataset with `cross_validate` (5-fold, 5 metrics each). Build a comparison table.

### Exercise 5 — Overfit vs Underfit
Plot learning curves for:
- `DecisionTreeClassifier(max_depth=1)` (underfit)
- `DecisionTreeClassifier(max_depth=None)` (overfit)
- `DecisionTreeClassifier(max_depth=5)` (good fit)

### Exercise 6 — Time Series Split
Simulate a time-series dataset (e.g., `sin(t) + noise`). Use `TimeSeriesSplit` and compare to plain K-Fold. Which is more honest for time-series?

---
## 🎤 12. Interview Questions

1. **What is a confusion matrix?**  
   A table comparing predicted vs actual classes — shows TP, TN, FP, FN.

2. **Difference between precision and recall?**  
   Precision = TP/(TP+FP) — trustworthiness of positive predictions. Recall = TP/(TP+FN) — coverage of actual positives.

3. **When would you prioritize recall over precision?**  
   When **false negatives are costly** — e.g., disease detection, fraud detection.

4. **What is ROC-AUC?**  
   Area under the ROC curve — measures ranking ability across all thresholds.

5. **ROC-AUC vs PR curve — when to use each?**  
   ROC-AUC for balanced data; PR curve for heavily imbalanced data.

6. **Why can accuracy mislead on imbalanced data?**  
   A lazy "predict majority class" model gets high accuracy but detects nothing.

7. **What is cross-validation?**  
   Repeatedly splitting data into train/val folds to get a more reliable performance estimate.

8. **Why is Stratified K-Fold useful for classification?**  
   It preserves class ratios in each fold — critical for imbalanced datasets.

9. **What is a learning curve?**  
   A plot of train and validation score vs training set size.

10. **How can you identify overfitting using a learning curve?**  
    High train score, low validation score, and a visible gap.

11. **What is data leakage?**  
    When info from outside the training set leaks into training, inflating scores.

12. **Why should the test set remain untouched until final evaluation?**  
    To preserve an honest final estimate of generalization.

---
## 🏁 13. Key Takeaways

- **Confusion Matrix** → understand TP, TN, FP, FN
- **Precision** → how reliable positive predictions are
- **Recall** → how many actual positives were found
- **F1** → balance between precision and recall
- **ROC-AUC** → ranking / separation across thresholds
- **PR Curve** → especially useful for imbalanced classification
- **Cross-Validation** → more reliable estimate of performance
- **Learning Curves** → diagnose underfitting and overfitting
- **Data Leakage** → can make evaluation look unrealistically good

### 🧭 Golden Rule

> **Always evaluate on unseen data. Choose the metric that matches the cost of your mistakes. When in doubt on imbalanced data, prefer PR-AUC over ROC-AUC.**

---

### 🔗 What's Next?

**Module 09 — Hyperparameter Tuning**

We'll cover **GridSearchCV**, **RandomizedSearchCV**, **Optuna**, and **Bayesian Optimization** — the tools that squeeze the last few percentage points out of your models.